In [2]:
import shutil
import os
import numpy as np
from ipie.analysis.extraction import extract_observable
from ipie.analysis.autocorr import reblock_by_autocorr  # Added for error analysis

# 1. Safely duplicate the file to bypass the active MPI write-lock
original_file = "estimates.0.h5"
temp_file = "estimates_temp.h5"

print(f"Reading active QMC data from {original_file}...\n")

try:
    shutil.copy(original_file, temp_file)
    
    # 2. Use the built-in ipie extractor
    qmc_data = extract_observable(temp_file, "energy")
    
    # 3. Extract the total energy array
    if "ETotal" in qmc_data:
        energies = np.array(qmc_data["ETotal"])
        blocks_done = len(energies)
        
        print(f"--- IPIE PROGRESS REPORT ---")
        print(f"Blocks Completed: {blocks_done} / 200")
        
        if blocks_done > 1:
            # --- THE STATISTICAL ERROR CALCULATION ---
            # reblock_by_autocorr handles the serial correlation between QMC blocks
            df_ac = reblock_by_autocorr(energies, verbose=0)
            
            # Extract the statistically independent mean and error
            mean_energy = float(df_ac["ETotal_ac"].iloc[0])
            error_bar = float(df_ac["ETotal_error_ac"].iloc[0])
            
            print(f"Latest Block:        {energies[-1]:.4f} Ha")
            print(f"Running Average:     {mean_energy:.6f} +/- {error_bar:.6f} Ha")
            
            # Print the last few blocks to see if it's stable
            tail = [round(float(e), 4) for e in energies[-min(200, blocks_done):]]
            print(f"Last 5 Blocks:       {tail}")
            
        elif blocks_done == 1:
            print(f"Current Energy:   {energies[-1]:.4f} Ha")
            print("(Waiting for more blocks to calculate error bars...)")
            
    else:
        print("Data extracted, but 'ETotal' column not found yet. Block 1 might still be running.")

except Exception as e:
    print(f"[!] Error extracting data: {e}")

finally:
    # Clean up the temp file
    if os.path.exists(temp_file):
        os.remove(temp_file)

Reading active QMC data from estimates.0.h5...

--- IPIE PROGRESS REPORT ---
Blocks Completed: 195 / 200
Latest Block:        -230.6546 Ha
Running Average:     -230.612192 +/- 0.013217 Ha
Last 5 Blocks:       [-230.1327, -230.2689, -230.3292, -230.3737, -230.4065, -230.4383, -230.4534, -230.4814, -230.4929, -230.5144, -230.5285, -230.5406, -230.5516, -230.5608, -230.5684, -230.5764, -230.5929, -230.6011, -230.5967, -230.6099, -230.5936, -230.5954, -230.5874, -230.5994, -230.6053, -230.6029, -230.5952, -230.6056, -230.6146, -230.6185, -230.6224, -230.6306, -230.6244, -230.6364, -230.6266, -230.6366, -230.6379, -230.6357, -230.6382, -230.6432, -230.6379, -230.6461, -230.6317, -230.6323, -230.6279, -230.6376, -230.6291, -230.6269, -230.6205, -230.6432, -230.6342, -230.6325, -230.6305, -230.6346, -230.6318, -230.6238, -230.6317, -230.633, -230.6419, -230.6365, -230.6228, -230.6235, -230.6213, -230.6184, -230.6254, -230.6204, -230.6225, -230.6348, -230.6296, -230.6423, -230.6479, -230.6383,